# Learn about hybrid search and graph search from gpt
# using pinecone vector db for this

In [3]:
!pip install --upgrade --quiet pinecone-client pinecone-text pinecone-notebooks


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
api_key=os.getenv("PINECONE_API_KEY")

In [6]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [8]:
from pinecone import Pinecone , ServerlessSpec
index_name="hybrid-search-langchain"
# initialize the client
pc=Pinecone(api_key=api_key)

# create the index if it does not exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, #384 because we are using sentence-transformer model for creating the embeddings which produces 384 dimensional embeddings, so this is basically for our vector search
        metric='dotproduct', #sparse values are only supported for dotproduct metric 
        spec=ServerlessSpec(cloud='aws',region='us-east-1')
    )

In [9]:
index=pc.Index(index_name)
index

c:\Users\DELL\OneDrive\Desktop\LangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# setting up vector embeddings and sparse matrix
from dotenv import load_dotenv

load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
from pinecone_text.sparse import BM25Encoder

bm25encoder=BM25Encoder().default()
bm25encoder
# responsible for encoding the text data into sparse vectors (uses tf-idf)

In [13]:
sentences=[
    'In 2023 , I visited Paris',
    'In 2022 , I visited New York',
    'In 2021 , I visited New Orleans'
]

# uses tfidf to encode the text data into sparse vectors, which are then used for sparse search in the pinecone index
bm25encoder.fit(sentences)

# store the values to a json file
bm25encoder.dump("bm25encoder.json")

100%|██████████| 3/3 [00:00<00:00, 10.02it/s]


In [14]:
retriever=PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25encoder,
    index=index
)

In [15]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x0000022ACC9A0810>, index=<pinecone.db_data.index.Index object at 0x0000022A9A4A5FD0>)

In [16]:
retriever.add_texts(sentences)

100%|██████████| 1/1 [00:05<00:00,  5.68s/it]


In [22]:
retriever.invoke('when did i visited new york?')

[Document(metadata={'score': 0.609650254}, page_content='In 2022 , I visited New York'),
 Document(metadata={'score': 0.344620556}, page_content='In 2021 , I visited New Orleans'),
 Document(metadata={'score': 0.213687569}, page_content='In 2023 , I visited Paris')]